# Phase 09 — MLOps & Drift Monitoring
## Sentinel AI: Enterprise Fraud Intelligence Platform

---

### The Scientific Question
Phase 9 shifts the focus from model building to operational reliability. The core question is:

> **Can Sentinel AI detect degradation early enough to maintain reliable fraud detection without unnecessary retraining?**

### MLOps Architecture
This notebook acts as the daily MLOps Orchestrator, executing a suite of autonomous drift engines:
1. **Feature Drift (PSI)**: Detects covariate shifts in the top 50 critical features.
2. **Prediction Drift**: Monitors sudden changes in the model's output distribution.
3. **Concept Drift**: Evaluates PR-AUC degradation (when ground truth settles).
4. **Embedding Drift**: Tracks geometric centroid shifts in the Graph Embeddings (Node2Vec).
5. **Alert Engine & Retraining Logic**: Maps statistical drift to business action without blindly auto-training.
6. **Shadow Deployment**: Silently evaluates a challenger model in the background.

In [1]:
from src.utils.bootstrap import bootstrap
bootstrap()

from pathlib import Path
import sys

import sys
from pathlib import Path

import numpy as np
import pandas as pd


from src.mlops.MLOpsOrchestrator import MLOpsOrchestrator
from src.graph_learning.model_registry.ModelRegistry import ModelRegistry, ModelRegistration

REPORTS_DIR = Path("reports/phase_09")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Sentinel AI MLOps Engine initialized.")

Sentinel AI MLOps Engine initialized.


## Section 1 — Model Registry & Provenance
Before monitoring, we track the Champion model's metadata in the Global Model Registry.

In [2]:
registry = ModelRegistry()

# Register the winning Phase 8 model
champ = registry.register_model(ModelRegistration(
    model_name="Full_Fusion_XGBoost",
    experiment_id="4_Full_Fusion",
    dataset_hash="1234abcd",
    metrics={"pr_auc": 0.95, "f1": 0.92},
    status="Champion"
))

print("=== Global Model Registry ===")
print(registry.list_models().to_string(index=False))

=== Global Model Registry ===
         model_name experiment_id dataset_hash                      metrics validation_status deployment_ready   status notes                           model_id    registration_date
Full_Fusion_XGBoost 4_Full_Fusion     1234abcd {'pr_auc': 0.95, 'f1': 0.92}              PASS              YES Champion       Full_Fusion_XGBoost_20260630211615 2026-06-30T21:16:15Z
Full_Fusion_XGBoost 4_Full_Fusion     1234abcd {'pr_auc': 0.95, 'f1': 0.92}              PASS              YES Champion       Full_Fusion_XGBoost_20260630211651 2026-06-30T21:16:51Z


## Section 2 — Simulate Production Data Drift
We load the reference dataset (Phase 8 fusion) and simulate a production day with covariate and concept drift.

In [3]:
# For demonstration, we'll generate synthetic baseline vs production data
np.random.seed(42)
N = 5000

df_ref = pd.DataFrame({
    "F100": np.random.normal(0, 1, N),
    "F200": np.random.normal(5, 2, N),
    "emb_0": np.random.normal(0.1, 0.05, N),
    "F3924": np.random.choice([0, 1], size=N, p=[0.95, 0.05]),
    "prediction": np.random.uniform(0, 1, N)
})

# Simulate Drift in Production
df_prod = df_ref.copy()
df_prod["F100"] = df_prod["F100"] + 0.8  # Strong covariate shift
df_prod["emb_0"] = df_prod["emb_0"] * 1.5  # Embedding variance shift
df_prod["prediction"] = df_prod["prediction"] + 0.1 # Prediction drift

top_features = ["F100", "F200"]
TARGET = "F3924"
PRED = "prediction"

print(f"Reference Dataset: {df_ref.shape}")
print(f"Production Dataset: {df_prod.shape}")

Reference Dataset: (5000, 5)
Production Dataset: (5000, 5)


## Section 3 — Daily MLOps Orchestrator
The Orchestrator calculates PSI, concept degradation, embedding shift, logs alerts, and recommends retraining.

In [4]:
orchestrator = MLOpsOrchestrator(output_dir=str(REPORTS_DIR))

orchestrator.run_daily_monitoring(
    df_ref=df_ref,
    df_prod=df_prod,
    top_features=top_features,
    target_col=TARGET,
    pred_col=PRED
)

Running daily MLOps monitoring pipeline...
- Checking Feature Drift (PSI)...
- Checking Prediction Drift...
- Checking Concept Drift (requires settled ground truth)...
- Checking Embedding Drift...
- Generating Alerts & Retraining Recommendations...
Pipeline complete. Status: IMMEDIATE_RETRAINING


## Section 4 — Reviewing the MLOps Reports
Let's inspect the alerts and the retraining recommendation generated by the Engine.

In [5]:
print("\n=== Alert Log ===")
try:
    alerts = pd.read_csv(REPORTS_DIR / "alert_log.csv")
    print(alerts.to_string(index=False))
except FileNotFoundError:
    print("No alerts triggered today.")

print("\n=== Retraining Recommendation ===")
with open(REPORTS_DIR / "retraining_recommendation.json", "r") as f:
    print(f.read())


=== Alert Log ===
           timestamp severity   category                   message  metric_value  threshold
2026-06-30T21:16:51Z CRITICAL Data Drift Extreme PSI shift in F100        0.6221        0.2

=== Retraining Recommendation ===
{
  "timestamp": "2026-06-30T21:16:51Z",
  "recommendation": "IMMEDIATE_RETRAINING",
  "reason": "Detected 1 CRITICAL alerts (e.g. Extreme PSI shift in F100).",
  "human_approval_required": true
}


## Section 5 — Shadow Deployment Simulation
A challenger model has been trained. Before replacing the Champion, it runs silently in Shadow Mode to verify superiority.

In [6]:
# Simulate Challenger predictions (slightly better than Champion)
y_true = df_prod[TARGET]
y_champ = df_prod[PRED]
y_chall = y_champ * 0.9 + (y_true * 0.1) # Artificially closer to truth

shadow_df = orchestrator.run_shadow_deployment(y_true, y_champ, y_chall)

print("\n=== Shadow Deployment Comparison ===")
print(shadow_df.to_string(index=False))

Running Shadow Deployment evaluation...

=== Shadow Deployment Comparison ===
     Model  PR-AUC     F1
  Champion  0.0516 0.0954
Challenger  0.1558 0.1169


## Section 6 — Phase 9 Summary

### Artifacts Generated
| Artifact | Location |
|---|---|
| `psi_report.csv` | `reports/phase_09/` |
| `prediction_drift.csv` | `reports/phase_09/` |
| `concept_drift.csv` | `reports/phase_09/` |
| `embedding_drift.json` | `reports/phase_09/` |
| `alert_log.csv` | `reports/phase_09/` |
| `retraining_recommendation.json` | `reports/phase_09/` |
| `shadow_comparison.csv` | `reports/phase_09/` |
| `model_registry.json` | `models/registry/` |

### Operational Conclusion
Sentinel AI is no longer a static notebook project. It is a **living enterprise system** capable of diagnosing its own degradation, escalating critical data shifts via Alert Engines, and systematically gating model updates behind Shadow Deployments and human approval.

With MLOps secured, we proceed to the final phase: **Phase 10: Production Deployment & Dashboard**.